In [1]:
import numpy as np
import pandas as pd
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


# Notebook 13 — Ensemble Methods

Hard voting · Soft voting · Stacking (LR meta-classifier)  
CPU-only. No GPU training. Resume-safe.

In [2]:
import os, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, accuracy_score, classification_report
warnings.filterwarnings("ignore")

IS_KAGGLE = os.path.exists("/kaggle/working")
REPO      = Path("/kaggle/working/Sarcasm_detection") if IS_KAGGLE else Path.cwd().parent

PREDS_DIR  = REPO / "04_outputs" / "predictions"
TABLES_DIR = REPO / "04_outputs" / "tables"
REPORTS_DIR = REPO / "04_outputs" / "reports"
TABLES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_CSV = TABLES_DIR / "13_ensemble_results.csv"

print(f"ENV         = {'kaggle' if IS_KAGGLE else 'local'}")
print(f"REPO        = {REPO}")
print(f"PREDS DIR   = {PREDS_DIR}")
print(f"RESULTS CSV = {RESULTS_CSV}")


ENV         = local
REPO        = /Users/sefayet/Desktop/Github/Sarcasm_detection
PREDS DIR   = /Users/sefayet/Desktop/Github/Sarcasm_detection/04_outputs/predictions
RESULTS CSV = /Users/sefayet/Desktop/Github/Sarcasm_detection/04_outputs/tables/13_ensemble_results.csv


## Prediction loading utilities

In [3]:
# Ordered longest-first to avoid substring collisions (banglasarc3 vs banglasarc)
DATASETS = ["banglasarc3", "banglasarc", "ben_sarc"]
TASKS    = ["ternary", "binary"]
# Both "val" and "validation" are normalised to "val" internally
SPLITS   = ["test", "val"]
SPLIT_ALIASES = {"validation": "val", "val": "val", "test": "test"}


def _col_search(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def load_preds(path):
    """Load prediction CSV -> (y_true, y_pred, prob_matrix).
    prob_matrix: (N, num_classes). Falls back to one-hot if no prob columns found."""
    df = pd.read_csv(path)

    label_col = _col_search(df, ["true_label", "label", "label_binary",
                                   "label_original", "ground_truth"])
    pred_col  = _col_search(df, ["pred_label", "pred", "predicted_label", "prediction"])

    if label_col is None or pred_col is None:
        raise ValueError(f"Cannot find label/pred columns in {path.name}. Cols: {list(df.columns)}")

    y_true = df[label_col].astype(int).values
    y_pred = df[pred_col].astype(int).values

    prob_cols = sorted([c for c in df.columns
                        if (c.startswith("prob_") or c.startswith("proba_") or c.startswith("score_"))
                        and c.split("_")[-1].isdigit()])
    if prob_cols:
        probs = df[prob_cols].values.astype(float)
    else:
        # no probability columns saved — one-hot encode predictions (hard voting still works)
        n_cls = int(y_true.max()) + 1
        probs = np.eye(n_cls)[y_pred]

    return y_true, y_pred, probs


def discover_registry(preds_dir):
    """Scan predictions directory.
    Returns: {(dataset, task): {model_id: {split: Path}}}
    Handles both _val_ / _validation_ and _preds / _predictions suffixes.
    model_id is everything in the filename before _{dataset}_{task}."""
    registry = {}

    # All raw split tokens we recognise (longer tokens first to avoid prefix clash)
    RAW_SPLITS = ["validation", "test", "val"]

    for fpath in sorted(preds_dir.glob("*.csv")):
        stem = fpath.stem  # filename without .csv

        # ── detect dataset ────────────────────────────────────────────────────
        ds = next((d for d in DATASETS if f"_{d}_" in stem or stem.endswith(f"_{d}")), None)
        if ds is None:
            continue

        # ── detect task ───────────────────────────────────────────────────────
        task = next((t for t in TASKS if f"_{t}_" in stem or stem.endswith(f"_{t}")), None)
        if task is None:
            continue

        # ── detect split (try _<split>_preds, _<split>_predictions, _<split>) ─
        raw_split = None
        for rs in RAW_SPLITS:
            if (f"_{rs}_preds"       in stem or
                f"_{rs}_predictions" in stem or
                stem.endswith(f"_{rs}_preds") or
                stem.endswith(f"_{rs}_predictions") or
                stem.endswith(f"_{rs}")):
                raw_split = rs
                break
        if raw_split is None:
            continue
        split = SPLIT_ALIASES[raw_split]   # normalise "validation" -> "val"

        # ── derive model_id ───────────────────────────────────────────────────
        cut = stem.find(f"_{ds}_{task}")
        model_id = stem[:cut] if cut != -1 else stem

        registry.setdefault((ds, task), {}).setdefault(model_id, {})[split] = fpath

    return registry

print("Prediction loading utilities ready.")


Prediction loading utilities ready.


## Ensemble functions

In [4]:
def hard_vote(pred_list, y_true):
    """Majority vote; ties broken by lowest class index."""
    votes = np.stack(pred_list, axis=1)       # (N, M)
    n_cls = int(y_true.max()) + 1
    counts = np.apply_along_axis(
        lambda row: np.bincount(row, minlength=n_cls), axis=1, arr=votes)
    y_hat = counts.argmax(axis=1)
    return (f1_score(y_true, y_hat, average="macro"),
            f1_score(y_true, y_hat, average="weighted"),
            accuracy_score(y_true, y_hat))


def soft_vote(prob_list, y_true):
    """Average probability distributions, then argmax."""
    avg   = np.mean(np.stack(prob_list, axis=0), axis=0)   # (N, C)
    y_hat = avg.argmax(axis=1)
    return (f1_score(y_true, y_hat, average="macro"),
            f1_score(y_true, y_hat, average="weighted"),
            accuracy_score(y_true, y_hat))


def stacking(val_probs, y_val, test_probs, y_test):
    """LR meta-learner trained on val probability stack, evaluated on test.
    val_probs / test_probs: lists of (N, C) arrays, one per model."""
    X_val  = np.concatenate(val_probs,  axis=1)
    X_test = np.concatenate(test_probs, axis=1)
    scaler = StandardScaler()
    X_val  = scaler.fit_transform(X_val)
    X_test = scaler.transform(X_test)
    lr = LogisticRegression(max_iter=1000, C=1.0, random_state=42, class_weight="balanced")
    lr.fit(X_val, y_val)
    y_hat_val  = lr.predict(X_val)
    y_hat_test = lr.predict(X_test)
    return (f1_score(y_test, y_hat_test, average="macro"),
            f1_score(y_test, y_hat_test, average="weighted"),
            accuracy_score(y_test, y_hat_test),
            f1_score(y_val,  y_hat_val,  average="macro"))   # val score

print("Ensemble functions ready.")


Ensemble functions ready.


## Discover available prediction files

In [5]:
registry = discover_registry(PREDS_DIR)

print(f"Prediction files found for {len(registry)} (dataset, task) combination(s):\n")
for (ds, task), models in sorted(registry.items()):
    complete = {mid for mid, splits in models.items()
                if "val" in splits and "test" in splits}
    print(f"  ({ds:12s}, {task:8s})  {len(complete):2d} complete model(s)")
    for mid in sorted(complete):
        print(f"    · {mid}")


Prediction files found for 4 (dataset, task) combination(s):

  (banglasarc  , binary  )  17 complete model(s)
    · 05_baseline_banglabert
    · 06_fgm_banglabert
    · 07_mbert
    · 07_muril
    · 07_sagorsarker-bb
    · 07_xlm-roberta
    · 10_label_smoothing
    · 10_rdrop
    · 10_rdrop_plus_ls
    · 11_supcon_joint
    · 11_supcon_twostage
    · 12_bb_fgm_e03
    · 12_bb_fgm_e08
    · 12_bb_fgm_e10
    · 12_bb_fgmawp
    · 12_bb_freelb_k3
    · 12_muril_fgm_e05
  (banglasarc3 , binary  )  17 complete model(s)
    · 05_baseline_banglabert
    · 06_fgm_banglabert
    · 07_mbert
    · 07_muril
    · 07_sagorsarker-bb
    · 07_xlm-roberta
    · 10_label_smoothing
    · 10_rdrop
    · 10_rdrop_plus_ls
    · 11_supcon_joint
    · 11_supcon_twostage
    · 12_bb_fgm_e03
    · 12_bb_fgm_e08
    · 12_bb_fgm_e10
    · 12_bb_fgmawp
    · 12_bb_freelb_k3
    · 12_muril_fgm_e05
  (banglasarc3 , ternary )  17 complete model(s)
    · 08_banglabert
    · 08_mbert
    · 08_muril
    · 08_sagorsar

## Main run loop (resume-safe)

In [6]:
SCHEMA = ["run_id", "ensemble_method", "pool_name", "dataset", "task",
          "num_models", "test_macro_f1", "test_weighted_f1", "test_accuracy", "val_macro_f1"]

if RESULTS_CSV.exists():
    res_df = pd.read_csv(RESULTS_CSV)
    done   = set(res_df["run_id"])
    print(f"Resuming: {len(done)} run(s) already in {RESULTS_CSV}")
else:
    res_df = pd.DataFrame(columns=SCHEMA)
    done   = set()
    print("Starting fresh.")


def _run_pool(ds, task, pool_name, pool_models):
    """Run hard_vote / soft_vote / stacking for one pool. Returns new result rows."""
    mids   = list(pool_models.keys())
    y_test = pool_models[mids[0]]["test_true"]
    y_val  = pool_models[mids[0]]["val_true"]
    rows   = []

    for method in ["hard_vote", "soft_vote", "stacking"]:
        run_id = f"{method}_{pool_name}_{ds}_{task}"
        if run_id in done:
            print(f"  [SKIP] {run_id}")
            continue

        try:
            if method == "hard_vote":
                tmf1, twf1, tacc = hard_vote([pool_models[m]["test_pred"] for m in mids], y_test)
                vmf1, *_         = hard_vote([pool_models[m]["val_pred"]  for m in mids], y_val)

            elif method == "soft_vote":
                tmf1, twf1, tacc = soft_vote([pool_models[m]["test_probs"] for m in mids], y_test)
                vmf1, *_         = soft_vote([pool_models[m]["val_probs"]  for m in mids], y_val)

            else:  # stacking
                tmf1, twf1, tacc, vmf1 = stacking(
                    [pool_models[m]["val_probs"]  for m in mids], y_val,
                    [pool_models[m]["test_probs"] for m in mids], y_test)

            row = dict(run_id=run_id, ensemble_method=method, pool_name=pool_name,
                       dataset=ds, task=task, num_models=len(mids),
                       test_macro_f1=round(tmf1, 4), test_weighted_f1=round(twf1, 4),
                       test_accuracy=round(tacc, 4), val_macro_f1=round(vmf1, 4))
            rows.append(row)
            done.add(run_id)
            print(f"  [DONE] {run_id:<60s}  test_mf1={tmf1:.4f}  val_mf1={vmf1:.4f}")

        except Exception as e:
            print(f"  [ERR ] {run_id}: {e}")

    return rows


# ── Main loop ─────────────────────────────────────────────────────────────────
for (ds, task), models in sorted(registry.items()):

    # Load all models that have both val and test splits
    loaded = {}
    for mid, splits in models.items():
        if "val" not in splits or "test" not in splits:
            continue
        try:
            vt, vp, vpr = load_preds(splits["val"])
            tt, tp, tpr = load_preds(splits["test"])
            val_mf1 = f1_score(vt, vp, average="macro")
            loaded[mid] = dict(val_true=vt, val_pred=vp, val_probs=vpr,
                               test_true=tt, test_pred=tp, test_probs=tpr,
                               val_macro_f1=val_mf1)
        except Exception as e:
            print(f"  LOAD WARN ({ds},{task}) {mid}: {e}")

    if len(loaded) < 2:
        print(f"\nSkipping ({ds},{task}): only {len(loaded)} loadable model(s).")
        continue

    ranked = sorted(loaded.items(), key=lambda x: x[1]["val_macro_f1"], reverse=True)
    print(f"\n{'='*70}")
    print(f"  ({ds}, {task})  —  {len(ranked)} models ranked by val macro-F1")
    print(f"{'='*70}")
    for mid, d in ranked:
        print(f"  {mid:55s}  val={d['val_macro_f1']:.4f}")

    r_ids = [m for m, _ in ranked]

    # ── Build pools ──────────────────────────────────────────────────────────
    pools = {}
    if len(r_ids) >= 3:
        pools["top_3"] = {m: loaded[m] for m in r_ids[:3]}
    if len(r_ids) >= 5:
        pools["top_5"] = {m: loaded[m] for m in r_ids[:5]}
    pools["all"] = loaded

    # diverse: best model per backbone family (ranked order preserves quality)
    seen_fam = {}
    for mid, _ in ranked:
        fam = ("muril"       if "muril"  in mid else
               "xlmr"        if "xlm"    in mid else
               "mbert"       if "mbert"  in mid else
               "sagorsarker" if "sagor"  in mid else
               "banglabert")
        if fam not in seen_fam:
            seen_fam[fam] = mid
    if len(seen_fam) >= 2:
        pools["diverse"] = {m: loaded[m] for m in seen_fam.values() if m in loaded}

    # ── Run each pool ─────────────────────────────────────────────────────────
    for pool_name, pool_models in pools.items():
        if len(pool_models) < 2:
            continue
        new_rows = _run_pool(ds, task, pool_name, pool_models)
        if new_rows:
            res_df = pd.concat([res_df, pd.DataFrame(new_rows)], ignore_index=True)
            res_df.to_csv(RESULTS_CSV, index=False)   # incremental save after every pool

print(f"\nTotal ensemble runs: {len(res_df)}")


Starting fresh.
  LOAD WARN (banglasarc,binary) 05_baseline_banglabert: Cannot find label/pred columns in 05_baseline_banglabert_banglasarc_binary_validation_predictions.csv. Cols: ['text', 'gold_label', 'pred_label', 'prob_label_0', 'prob_label_1', 'correct', 'dataset', 'split', 'model_name', 'seed', 'max_length', 'epochs', 'learning_rate', 'use_normalization']

  (banglasarc, binary)  —  16 models ranked by val macro-F1
  12_bb_fgm_e08                                            val=0.9917
  12_bb_fgm_e03                                            val=0.9896
  12_bb_fgm_e10                                            val=0.9896
  12_bb_fgmawp                                             val=0.9896
  06_fgm_banglabert                                        val=0.9896
  10_rdrop_plus_ls                                         val=0.9855
  11_supcon_twostage                                       val=0.9834
  11_supcon_joint                                          val=0.9834
  12_bb_freelb

## Summary tables

In [7]:
# ── Best single-model results per dataset/task (from nb05-12) ─────────────────
BEST_SINGLE = {
    ("banglasarc",  "binary"):  ("bb_fgmawp / bb_fgm_e05 (nb06/12)", 0.9896),
    ("banglasarc3", "binary"):  ("bb_fgm_e05 (nb06)",                 0.7880),
    ("banglasarc3", "ternary"): ("bb_fgm_e03 (nb12)",                 0.6618),
    ("ben_sarc",    "binary"):  ("bb_fgmawp (nb12)",                  0.8126),
}

# ── TABLE 1 — Best ensemble vs best single model ──────────────────────────────
SEP = "=" * 90
print(SEP)
print("  TABLE 1 — Best Ensemble vs Best Single Model  (Test Macro-F1)")
print(SEP)

t1_rows = []
for (ds, task), (single_name, single_f1) in sorted(BEST_SINGLE.items()):
    sub = res_df[(res_df.dataset == ds) & (res_df.task == task)]
    if sub.empty:
        continue
    best = sub.loc[sub.test_macro_f1.idxmax()]
    delta = round(best.test_macro_f1 - single_f1, 4)
    t1_rows.append({
        "dataset_task":     f"{ds}_{task}",
        "best_single_f1":   single_f1,
        "best_single_model":single_name,
        "best_ensemble_f1": best.test_macro_f1,
        "best_ensemble":    f"{best.ensemble_method} / {best.pool_name}",
        "delta":            delta,
        "ensemble_wins":    "YES" if delta > 0 else "NO",
    })

t1 = pd.DataFrame(t1_rows)
if not t1.empty:
    pd.set_option("display.max_colwidth", 50)
    pd.set_option("display.width", 200)
    print(t1.to_string(index=False))

# ── TABLE 2 — Method comparison ───────────────────────────────────────────────
print("\n" + SEP)
print("  TABLE 2 — Ensemble Method Comparison  (Test Macro-F1)")
print(SEP)

t2 = (res_df.groupby("ensemble_method")["test_macro_f1"]
            .agg(["mean", "max", "min", "count"])
            .round(4)
            .sort_values("mean", ascending=False)
            .rename(columns={"mean":"mean_mf1", "max":"max_mf1",
                              "min":"min_mf1", "count":"n_runs"}))
print(t2.to_string())

# ── TABLE 3 — Pool comparison ─────────────────────────────────────────────────
print("\n" + SEP)
print("  TABLE 3 — Pool Comparison  (Test Macro-F1)")
print(SEP)

t3 = (res_df.groupby("pool_name")["test_macro_f1"]
            .agg(["mean", "max", "count"])
            .round(4)
            .sort_values("mean", ascending=False)
            .rename(columns={"mean":"mean_mf1", "max":"max_mf1", "count":"n_runs"}))
print(t3.to_string())


  TABLE 1 — Best Ensemble vs Best Single Model  (Test Macro-F1)
       dataset_task  best_single_f1                best_single_model  best_ensemble_f1      best_ensemble   delta ensemble_wins
  banglasarc_binary          0.9896 bb_fgmawp / bb_fgm_e05 (nb06/12)            0.9897 stacking / diverse  0.0001           YES
 banglasarc3_binary          0.7880                bb_fgm_e05 (nb06)            0.7680   stacking / top_3 -0.0200            NO
banglasarc3_ternary          0.6618                bb_fgm_e03 (nb12)            0.6664    hard_vote / all  0.0046           YES
    ben_sarc_binary          0.8126                 bb_fgmawp (nb12)            0.8186    soft_vote / all  0.0060           YES

  TABLE 2 — Ensemble Method Comparison  (Test Macro-F1)
                 mean_mf1  max_mf1  min_mf1  n_runs
ensemble_method                                    
stacking           0.8037   0.9897   0.6431      16
hard_vote          0.7982   0.9855   0.6347      16
soft_vote          0.7976   0.9

## Grand ranking — all techniques across all notebooks

In [8]:
SEP = "=" * 70
print(SEP)
print("  GRAND RANKING — All Techniques  (mean Test Macro-F1)")
print(SEP)

# Fixed results from nb04-12 (binary datasets; nb05/06 mean over 3 binary tasks)
prior_results = [
    ("BanglaBERT+FGM eps=0.5 (nb06)",       0.8599),
    ("BanglaBERT baseline (nb05)",           0.8506),
    ("MuRIL (nb07)",                         0.8296),
    ("XLM-RoBERTa (nb07)",                   0.8122),
    ("Sagorsarker BB (nb07)",                0.8094),
    ("mBERT (nb07)",                         0.8075),
    ("BanglaBERT+FGM best-eps (nb12)",       0.8030),
    ("SupCon Joint (nb11)",                  0.8001),
    ("BanglaBERT+FGM+AWP (nb12)",            0.7990),
    ("BanglaBERT+FreeLB K=3 (nb12)",         0.7962),
    ("R-Drop+LS (nb10)",                     0.7960),
    ("MuRIL+FGM eps=0.5 (nb12)",             0.7782),
    ("TF-IDF + Logistic Regression (nb04)",  0.7386),
]

# Best ensemble per method from this notebook (mean across all runs)
ensemble_results = []
for method in ["soft_vote", "hard_vote", "stacking"]:
    sub = res_df[res_df.ensemble_method == method]
    if sub.empty:
        continue
    mean_f1 = round(sub.test_macro_f1.mean(), 4)
    best_row = sub.loc[sub.test_macro_f1.idxmax()]
    label = f"{method} best ensemble (nb13)"
    ensemble_results.append((label, mean_f1))

all_results = prior_results + ensemble_results
grand_df = (pd.DataFrame(all_results, columns=["method", "mean_macro_f1"])
              .sort_values("mean_macro_f1", ascending=False)
              .reset_index(drop=True))
grand_df.index += 1
print(grand_df.to_string())

print(f"\n{'='*70}")
print("  NOTEBOOK 13 COMPLETE")
print(f"{'='*70}")
print("Outputs:")
print(f"  04_outputs/tables/13_ensemble_results.csv")
print(f"  04_outputs/tables/13_table1_best_vs_single.csv")
print(f"  04_outputs/tables/13_table2_method_comparison.csv")
print(f"  04_outputs/tables/13_grand_ranking.csv")
print(f"\nNext -> Notebook 14: Cross-Dataset Comprehensive Evaluation")


  GRAND RANKING — All Techniques  (mean Test Macro-F1)
                                 method  mean_macro_f1
1         BanglaBERT+FGM eps=0.5 (nb06)         0.8599
2            BanglaBERT baseline (nb05)         0.8506
3                          MuRIL (nb07)         0.8296
4                    XLM-RoBERTa (nb07)         0.8122
5                 Sagorsarker BB (nb07)         0.8094
6                          mBERT (nb07)         0.8075
7         stacking best ensemble (nb13)         0.8037
8        BanglaBERT+FGM best-eps (nb12)         0.8030
9                   SupCon Joint (nb11)         0.8001
10            BanglaBERT+FGM+AWP (nb12)         0.7990
11       hard_vote best ensemble (nb13)         0.7982
12       soft_vote best ensemble (nb13)         0.7976
13         BanglaBERT+FreeLB K=3 (nb12)         0.7962
14                     R-Drop+LS (nb10)         0.7960
15             MuRIL+FGM eps=0.5 (nb12)         0.7782
16  TF-IDF + Logistic Regression (nb04)         0.7386

  NOTEBOO

## Save all outputs

In [9]:
res_df.to_csv(RESULTS_CSV, index=False)

t1_path    = TABLES_DIR / "13_table1_best_vs_single.csv"
t2_path    = TABLES_DIR / "13_table2_method_comparison.csv"
grand_path = TABLES_DIR / "13_grand_ranking.csv"

if not t1.empty: t1.to_csv(t1_path, index=False)
t2.to_csv(t2_path)
t3.to_csv(TABLES_DIR / "13_table3_pool_comparison.csv")
grand_df.to_csv(grand_path, index=False)

print("All artifacts saved.\n")
print("Files produced:")
for p in [RESULTS_CSV, t1_path, t2_path, grand_path]:
    if p.exists():
        kb = p.stat().st_size / 1024
        print(f"  {str(p.relative_to(REPO))}  ({kb:.1f} KB)")


All artifacts saved.

Files produced:
  04_outputs/tables/13_ensemble_results.csv  (4.7 KB)
  04_outputs/tables/13_table1_best_vs_single.csv  (0.4 KB)
  04_outputs/tables/13_table2_method_comparison.csv  (0.1 KB)
  04_outputs/tables/13_grand_ranking.csv  (0.5 KB)
